In [ ]:
!pip install torch torchvision tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import STL10
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import os


In [ ]:
class SimCLRAugment:
    def __init__(self, size=96):
        self.transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0)),
            T.ToTensor(),
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)


In [ ]:
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

class SimCLRDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        x1, x2 = img
        return x1, x2

base = STL10(root=DATA_DIR, split="unlabeled", download=True, transform=SimCLRAugment())
train_dataset = SimCLRDataset(base)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)


100%|██████████| 2.64G/2.64G [00:34<00:00, 75.8MB/s]


In [ ]:
class SimCLR(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(base.children())[:-1])  # remove FC
        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, projection_dim)
        )

    def forward(self, x):
        feats = self.encoder(x).flatten(1)
        proj = self.projector(feats)
        return F.normalize(proj, dim=1)

# Contrastive Loss
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.T) / temperature
    mask = ~torch.eye(2*N, dtype=bool, device=z.device)
    sim = sim.masked_select(mask).view(2*N, 2*N-1)

    positives = (z1 * z2).sum(dim=-1) / temperature
    positives = torch.cat([positives, positives], dim=0)

    loss = -torch.log(torch.exp(positives) / torch.sum(torch.exp(sim), dim=1))
    return loss.mean()


In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/ssl_checkpoints/simclr"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RESUME_FROM = "/content/drive/MyDrive/ssl_checkpoints/simclr/simclr_epoch_40.pth"  # 👉 change to path like "/content/checkpoints/simclr_epoch_20.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimCLR().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scaler = torch.cuda.amp.GradScaler()  # AMP scaler

start_epoch = 41

if RESUME_FROM is not None:
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed training from epoch {start_epoch}")


/tmp/ipython-input-1161708046.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # AMP scaler


Resumed training from epoch 41


In [ ]:
EPOCHS = 65  # change as needed

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    total_loss = 0

    for (x1, x2) in loop:
        x1, x2 = x1.to(device), x2.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            z1 = model(x1)
            z2 = model(x2)
            loss = nt_xent_loss(z1, z2)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg = total_loss / len(train_loader)
    print(f"Epoch {epoch} Completed. Avg Loss = {avg:.4f}")

    # SAVE CHECKPOINT every 5 epochs
    if epoch % 5 == 0:
        ckpt_path = f"{CHECKPOINT_DIR}/simclr_epoch_{epoch}.pth"
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict()
        }, ckpt_path)
        print("💾 Saved:", ckpt_path)


Epoch 41/65:   0%|          | 0/390 [00:00<?, ?it/s]/tmp/ipython-input-3371459197.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 41/65: 100%|██████████| 390/390 [09:28<00:00,  1.46s/it, loss=4.58]


Epoch 41 Completed. Avg Loss = 4.5865


Epoch 42/65: 100%|██████████| 390/390 [09:31<00:00,  1.47s/it, loss=4.58]


Epoch 42 Completed. Avg Loss = 4.5846


Epoch 43/65: 100%|██████████| 390/390 [09:23<00:00,  1.44s/it, loss=4.57]


...

Epoch 63 Completed. Avg Loss = 4.5581


Epoch 64/65: 100%|██████████| 390/390 [09:15<00:00,  1.43s/it, loss=4.54]


Epoch 64 Completed. Avg Loss = 4.5555


Epoch 65/65: 100%|██████████| 390/390 [09:21<00:00,  1.44s/it, loss=4.57]


Epoch 65 Completed. Avg Loss = 4.5545
💾 Saved: /content/drive/MyDrive/ssl_checkpoints/simclr/simclr_epoch_65.pth
